# Reproducible example of failure to rechunk into fsspec.mapping.FSMap object

But it works with a `zarr.storage.FSStore` object?

How I got here:
https://github.com/pangeo-data/rechunker/issues/137
The bit about using FSStore i only got from the tests here: https://github.com/pangeo-data/rechunker/pull/136/files its nowhere in the docs (https://rechunker.readthedocs.io/en/latest/tutorial.html#Cloud-Example)

Where are people supposed to learn about this? Should `fs.get_mapper()` show a warning?



In [1]:
# ! pip install "rechunker>0.5.0" # install rechunker from main to get the fix for this https://github.com/pangeo-data/rechunker/issues/135
# this does not work?

In [2]:
from scale_aware_air_sea.utils import open_zarr
# from scale_aware_air_sea.cm26_utils import load_and_combine_cm26
from scale_aware_air_sea.parameters import get_params
import numpy as np
import gcsfs
import matplotlib.pyplot as plt
import zarr
from rechunker import rechunk

import cftime
import xarray as xr

In [3]:
import rechunker
rechunker.__version__

'0.5.1'

In [4]:
from distributed import Client, LocalCluster
cluster = LocalCluster(n_workers=4, threads_per_worker=4)
client = Client(cluster)
client

Connection method: Cluster object,Cluster type: distributed.LocalCluster
Dashboard: /user/jbusecke/proxy/8787/status,
Dashboard: /user/jbusecke/proxy/8787/status,Workers: 4
Total threads: 16,Total memory: 125.81 GiB
Status: running,Using processes: True
Comm: tcp://127.0.0.1:38699,Workers: 4
Dashboard: /user/jbusecke/proxy/8787/status,Total threads: 16
Started: Just now,Total memory: 125.81 GiB
Comm: tcp://127.0.0.1:46687,Total threads: 4
Dashboard: /user/jbusecke/proxy/39359/status,Memory: 31.45 GiB
Nanny: tcp://127.0.0.1:35199,


In [7]:
# load global parameters
params = get_params('v0.7.0', test=False) 

fs = gcsfs.GCSFileSystem()
# import fsspec
# fs = fsspec.filesystem('gcs')

mapper_online_flux = fs.get_mapper('gs://cmip6/GFDL_CM2_6/control/ocean_boundary')
mapper_offline_flux = fs.get_mapper(params['paths']['CM26']['filter_fluxes'])
mapper_ice_temp = fs.get_mapper(params['paths']['CM26']['ice_mask_temp'])
mapper_ice_rechunked = fs.get_mapper(params['paths']['CM26']['ice_mask_rechunked'])
mapper_ice = fs.get_mapper(params['paths']['CM26']['ice_mask'])

ds_online = open_zarr(mapper_online_flux)
ds_offline = open_zarr(mapper_offline_flux)

In [1]:
mapper_ice

NameError: name 'mapper_ice' is not defined

## Pad the edges of the melt array

In [8]:
# creating a time resolved ice mask
melt = ds_online.melt

melt_padded = melt.pad(pad_width={'time':1}, mode='edge')
# replace the time with properly padded values
padded_time = np.hstack([
    np.array([cftime.DatetimeJulian(180, 12, 15, 12, 0, 0)]),
    melt.time.data,
    np.array([cftime.DatetimeJulian(201, 1, 16, 12, 0, 0)])
])
melt_padded = melt_padded.assign_coords(time=padded_time)
melt_padded

<xarray.DataArray 'melt' (time: 242, yt_ocean: 2700, xt_ocean: 3600)>
dask.array<concatenate, shape=(242, 2700, 3600), dtype=float32, chunksize=(1, 2700, 3600), chunktype=numpy.ndarray>
Coordinates:
    geolat_t  (yt_ocean, xt_ocean) float32 dask.array<chunksize=(338, 450), meta=np.ndarray>
    geolon_t  (yt_ocean, xt_ocean) float32 dask.array<chunksize=(338, 450), meta=np.ndarray>
  * time      (time) object 0180-12-15 12:00:00 ... 0201-01-16 12:00:00
  * xt_ocean  (xt_ocean) float64 -279.9 -279.8 -279.7 ... 79.75 79.85 79.95
  * yt_ocean  (yt_ocean) float64 -81.11 -81.07 -81.02 ... 89.89 89.94 89.98
Attributes:
    cell_methods:   time: mean
    long_name:      water flux transferred with sea ice form/melt (>0 enters ...
    standard_name:  water_flux_into_sea_water_due_to_sea_ice_thermodynamics
    time_avg_info:  average_T1,average_T2,average_DT
    units:          (kg/m^3)*(m/sec)
    valid_range:    [-1000000.0, 1000000.0]

## Rechunk

In [9]:
from collections.abc import MutableMapping
isinstance(mapper_ice_rechunked, MutableMapping)

True

In [12]:
from zarr.storage import FSStore

In [13]:
import zarr
zarr.__version__

'2.13.6'

In [13]:
'gs://'+mapper_ice_rechunked.root

'gs://leap-scratch/jbusecke/scale-aware-air-sea/temp/ice_mask_rechunked_CM26_v0.7.0.zarr'

In [14]:
rechunked = rechunk(
    melt.to_dataset(name='melt'),
    target_chunks={'time':242, 'xt_ocean':2},
    target_store=FSStore('gs://'+mapper_ice_rechunked.root),
    temp_store=FSStore('gs://'+mapper_ice_temp.root),
    max_mem="32GB"
)

In [15]:
rechunked

<Rechunked>
* Source      : <xarray.Dataset>
Dimensions:   (yt_ocean: 2700, xt_ocean: 3600, time: 240)
Coordinates:
    geolat_t  (yt_ocean, xt_ocean) float32 dask.array<chunksize=(338, 450), meta=np.ndarray>
    geolon_t  (yt_ocean, xt_ocean) float32 dask.array<chunksize=(338, 450), meta=np.ndarray>
  * time      (time) object 0181-01-16 12:00:00 ... 0200-12-16 12:00:00
  * xt_ocean  (xt_ocean) float64 -279.9 -279.8 -279.7 ... 79.75 79.85 79.95
  * yt_ocean  (yt_ocean) float64 -81.11 -81.07 -81.02 ... 89.89 89.94 89.98
Data variables:
    melt      (time, yt_ocean, xt_ocean) float32 dask.array<chunksize=(1, 2700, 3600), meta=np.ndarray>

* Intermediate: <zarr.hierarchy.Group '/'>

* Target      : <zarr.hierarchy.Group '/'>

In [16]:
rechunked.execute()

<zarr.hierarchy.Group '/'>

In [16]:
mapper_ice_rechunked.root

'leap-scratch/jbusecke/scale-aware-air-sea/temp/ice_mask_rechunked_CM26_v0.7.0.zarr'

In [17]:
fs.ls('leap-scratch/jbusecke/scale-aware-air-sea/temp/')

['leap-scratch/jbusecke/scale-aware-air-sea/temp/CESM_v0.6.2test.zarr',
 'leap-scratch/jbusecke/scale-aware-air-sea/temp/CESM_v0.7.0test.zarr',
 'leap-scratch/jbusecke/scale-aware-air-sea/temp/ice_mask_rechunked_CM26_v0.7.0.zarr',
 'leap-scratch/jbusecke/scale-aware-air-sea/temp/ice_mask_temp_CM26_v0.7.0.zarr']

In [20]:
ds_test = xr.open_dataset(mapper_ice_rechunked, engine='zarr', chunks={})

/tmp/ipykernel_2501/760066796.py:1: RuntimeWarning: Failed to open Zarr store with consolidated metadata, but successfully read with non-consolidated metadata. This is typically much slower for opening a dataset. To silence this warning, consider:
1. Consolidating metadata in this existing store with zarr.consolidate_metadata().
2. Explicitly setting consolidated=False, to avoid trying to read consolidate metadata, or
3. Explicitly setting consolidated=True, to raise an error in this case instead of falling back to try reading non-consolidated metadata.
  ds_test = xr.open_dataset(mapper_ice_rechunked, engine='zarr', chunks={})


In [21]:
ds_test

<xarray.Dataset>
Dimensions:   (yt_ocean: 2700, xt_ocean: 3600, time: 240)
Coordinates:
    geolat_t  (yt_ocean, xt_ocean) float32 dask.array<chunksize=(338, 2), meta=np.ndarray>
    geolon_t  (yt_ocean, xt_ocean) float32 dask.array<chunksize=(338, 2), meta=np.ndarray>
  * time      (time) object 0181-01-16 12:00:00 ... 0200-12-16 12:00:00
  * xt_ocean  (xt_ocean) float64 -279.9 -279.8 -279.7 ... 79.75 79.85 79.95
  * yt_ocean  (yt_ocean) float64 -81.11 -81.07 -81.02 ... 89.89 89.94 89.98
Data variables:
    melt      (time, yt_ocean, xt_ocean) float32 dask.array<chunksize=(240, 2700, 2), meta=np.ndarray>

In [5]:
# melt_resampled = melt_padded.resample(time='24H', base=12).interpolate().sel(time=slice('0181', '0201')).chunk({'time':3})
# # join with the actual daily data
# melt_resampled,_ = xr.align(melt_resampled, ds_offline, join='inner')


# melt_resampled = melt_padded.interp(time=ds_offline.time.chunk({'time':3}))
# ice_mask = abs(melt_resampled)>0
# ice_mask

# melt_padded

<xarray.DataArray 'melt' (time: 242, yt_ocean: 2700, xt_ocean: 3600)>
dask.array<concatenate, shape=(242, 2700, 3600), dtype=float32, chunksize=(1, 2700, 3600), chunktype=numpy.ndarray>
Coordinates:
    geolat_t  (yt_ocean, xt_ocean) float32 dask.array<chunksize=(338, 450), meta=np.ndarray>
    geolon_t  (yt_ocean, xt_ocean) float32 dask.array<chunksize=(338, 450), meta=np.ndarray>
  * time      (time) object 0180-12-15 12:00:00 ... 0201-01-16 12:00:00
  * xt_ocean  (xt_ocean) float64 -279.9 -279.8 -279.7 ... 79.75 79.85 79.95
  * yt_ocean  (yt_ocean) float64 -81.11 -81.07 -81.02 ... 89.89 89.94 89.98
Attributes:
    cell_methods:   time: mean
    long_name:      water flux transferred with sea ice form/melt (>0 enters ...
    standard_name:  water_flux_into_sea_water_due_to_sea_ice_thermodynamics
    time_avg_info:  average_T1,average_T2,average_DT
    units:          (kg/m^3)*(m/sec)
    valid_range:    [-1000000.0, 1000000.0]

In [5]:
melt_resampled = melt.resample(time='24H', base=12).interpolate() # .sel(time=slice('0181', '0201')).chunk({'time':3})

In [7]:
melt_interpolated = melt.interp(time=ds_offline.time)

In [9]:
melt

<xarray.DataArray 'melt' (time: 240, yt_ocean: 2700, xt_ocean: 3600)>
dask.array<open_dataset-bb5c7432586382659ea5c140b5a53c5cmelt, shape=(240, 2700, 3600), dtype=float32, chunksize=(1, 2700, 3600), chunktype=numpy.ndarray>
Coordinates:
    geolat_t  (yt_ocean, xt_ocean) float32 dask.array<chunksize=(338, 450), meta=np.ndarray>
    geolon_t  (yt_ocean, xt_ocean) float32 dask.array<chunksize=(338, 450), meta=np.ndarray>
  * time      (time) object 0181-01-16 12:00:00 ... 0200-12-16 12:00:00
  * xt_ocean  (xt_ocean) float64 -279.9 -279.8 -279.7 ... 79.75 79.85 79.95
  * yt_ocean  (yt_ocean) float64 -81.11 -81.07 -81.02 ... 89.89 89.94 89.98
Attributes:
    cell_methods:   time: mean
    long_name:      water flux transferred with sea ice form/melt (>0 enters ...
    standard_name:  water_flux_into_sea_water_due_to_sea_ice_thermodynamics
    time_avg_info:  average_T1,average_T2,average_DT
    units:          (kg/m^3)*(m/sec)
    valid_range:    [-1000000.0, 1000000.0]

In [8]:
melt_interpolated

<xarray.DataArray 'melt' (time: 7305, yt_ocean: 2700, xt_ocean: 3600)>
dask.array<transpose, shape=(7305, 2700, 3600), dtype=float32, chunksize=(7305, 2700, 3600), chunktype=numpy.ndarray>
Coordinates:
    geolat_t  (yt_ocean, xt_ocean) float32 dask.array<chunksize=(338, 450), meta=np.ndarray>
    geolon_t  (yt_ocean, xt_ocean) float32 dask.array<chunksize=(338, 450), meta=np.ndarray>
  * xt_ocean  (xt_ocean) float64 -279.9 -279.8 -279.7 ... 79.75 79.85 79.95
  * yt_ocean  (yt_ocean) float64 -81.11 -81.07 -81.02 ... 89.89 89.94 89.98
  * time      (time) object 0181-01-01 12:00:00 ... 0200-12-31 12:00:00
Attributes:
    cell_methods:   time: mean
    long_name:      water flux transferred with sea ice form/melt (>0 enters ...
    standard_name:  water_flux_into_sea_water_due_to_sea_ice_thermodynamics
    time_avg_info:  average_T1,average_T2,average_DT
    units:          (kg/m^3)*(m/sec)
    valid_range:    [-1000000.0, 1000000.0]

In [11]:
melt_resampled.isel(time=0).plot()

2023-03-16 18:26:10,209 - distributed.worker.memory - WARNING - Unmanaged memory use is high. This may indicate a memory leak or the memory may not be released to the OS; see https://distributed.dask.org/en/latest/worker-memory.html#memory-not-released-back-to-the-os for more information. -- Unmanaged memory: 24.59 GiB -- Worker memory limit: 31.45 GiB
2023-03-16 18:26:10,242 - distributed.worker.memory - WARNING - Unmanaged memory use is high. This may indicate a memory leak or the memory may not be released to the OS; see https://distributed.dask.org/en/latest/worker-memory.html#memory-not-released-back-to-the-os for more information. -- Unmanaged memory: 24.60 GiB -- Worker memory limit: 31.45 GiB
2023-03-16 18:26:10,342 - distributed.worker.memory - WARNING - Unmanaged memory use is high. This may indicate a memory leak or the memory may not be released to the OS; see https://distributed.dask.org/en/latest/worker-memory.html#memory-not-released-back-to-the-os for more information. 